# Prepare LM

In [ ]:
!pip install transformers torchaudio pyctcdecode datasets arpa evaluate
# !pip install https://github.com/kpu/kenlm/archive/master.zip

In [ ]:
!git clone https://github.com/kpu/kenlm.git

Cloning into 'kenlm'...
remote: Enumerating objects: 14170, done.
remote: Counting objects: 100% (123/123), done.
remote: Compressing objects: 100% (67/67), done.
remote: Total 14170 (delta 85), reused 56 (delta 56), pack-reused 14047 (from 4)
Receiving objects: 100% (14170/14170), 5.98 MiB | 18.95 MiB/s, done.
Resolving deltas: 100% (8041/8041), done.


In [ ]:
%cd /content/kenlm
!pwd
!mkdir -p build
%cd build
!cmake ..
!make -j 4

%cd ../..

/content/kenlm
/content/kenlm
/content/kenlm/build
CMake Deprecation Warning at CMakeLists.txt:1 (cmake_minimum_required):
  Compatibility with CMake < 3.10 will be removed from a future version of
  CMake.

  Update the VERSION argument <min> value.  Or, use the <min>...<max> syntax
  to tell CMake that the project requires at least <min> but has been updated
  to work with policies introduced by <max> or earlier.


-- The C compiler identification is GNU 11.4.0
-- The CXX compiler identification is GNU 11.4.0
-- Detecting C compiler ABI info
-- Detecting C compiler ABI info - done
-- Check for working C compiler: /usr/bin/cc - skipped
-- Detecting C compile features
-- Detecting C compile features - done
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done
-- Check for working CXX compiler: /usr/bin/c++ - skipped
-- Detecting CXX compile features
-- Detecting CXX compile features - done
-- Could NOT find Eigen3 (missing: Eigen3_DIR)
CMake Warning (dev) at CMak

In [ ]:
!pip install pythainlp deepcut
# !pip install tqdm

In [ ]:
# https://drive.google.com/file/d/1znJaRc3C1fqDzvwtkS4LFl4W_VjCAATG/view?usp=drive_link
# /content/gdrive/MyDrive/material project sm/lst20_corpus.zip
!unzip 'Your-dataset-folder-url-here'

Archive:  /content/gdrive/MyDrive/material project sm/lst20_corpus.zip
   creating: LST20_Corpus/
  inflating: LST20_Corpus/AGREEMENT.txt  
  inflating: LST20_Corpus/DESCRIPTION.txt  
   creating: LST20_Corpus/eval/
  inflating: LST20_Corpus/eval/T01608.txt  
  inflating: LST20_Corpus/eval/T01738.txt  
  inflating: LST20_Corpus/eval/T01739.txt  
  inflating: LST20_Corpus/eval/T01740.txt  
  inflating: LST20_Corpus/eval/T10077.txt  
  inflating: LST20_Corpus/eval/T10200.txt  
  inflating: LST20_Corpus/eval/T10218.txt  
  inflating: LST20_Corpus/eval/T10219.txt  
  inflating: LST20_Corpus/eval/T10259.txt  
  inflating: LST20_Corpus/eval/T10541.txt  
  inflating: LST20_Corpus/eval/T10572.txt  
  inflating: LST20_Corpus/eval/T10854.txt  
  inflating: LST20_Corpus/eval/T11130.txt  
  inflating: LST20_Corpus/eval/T11195.txt  
  inflating: LST20_Corpus/eval/T11389.txt  
  inflating: LST20_Corpus/eval/T11510.txt  
  inflating: LST20_Corpus/eval/T11518.txt  
  inflating: LST20_Corpus/eval/T1151

In [ ]:
!cat LST20_Corpus/train/T*.txt > LST20_Corpus/train.txt
!cat LST20_Corpus/eval/T*.txt > LST20_Corpus/val.txt
!cat LST20_Corpus/test/T*.txt > LST20_Corpus/test.txt

In [ ]:
!cat LST20_Corpus/*.txt > LST20_Corpus/text.txt

In [ ]:
from pythainlp.tokenize import word_tokenize
from tqdm import tqdm

# def segment_text(input_file, output_file):
#     with open(output_file, "w", encoding='utf-8') as fw:
#         with open(input_file, encoding='utf-8') as fp:
#             lines = fp.readlines()

#             # ใช้ tqdm เพื่อแสดง progress bar
#             for line in tqdm(lines, desc="Processing", unit="line"):
#                 words = word_tokenize(line, engine="attacut")
#                 fw.write(" ".join(words) + "\n")

def segment_text(input_file, output_file):
    with open(output_file, "w", encoding='utf-8') as fw:
        with open(input_file, encoding='utf-8') as fp:
            lines = fp.readlines()

            # ใช้ tqdm เพื่อแสดง progress bar
            for line in tqdm(lines, desc="Processing", unit="line"):
                fw.write(line.strip() + "\n")



In [ ]:
segment_text("/content/LST20_Corpus/text.txt", "segmented_text.txt")

Processing: 100%|██████████| 3241843/3241843 [00:01<00:00, 2459457.41line/s]


In [ ]:
import csv

# อ่านไฟล์ CSV แล้วเขียนเป็นไฟล์ TXT
with open('thai-government-corpus.csv', 'r', encoding='utf-8') as csv_file, open('segmented_text.txt', 'w', encoding='utf-8') as txt_file:
    csv_reader = csv.reader(csv_file)

    for row in csv_reader:
        txt_file.write("\t".join(row) + "\n")


In [ ]:
!kenlm/build/bin/lmplz -o 3 < segmented_text.txt > 3_gram_th.arpa

=== 1/5 Counting and sorting n-grams ===
Reading /content/segmented_text.txt
----5---10---15---20---25---30---35---40---45---50---55---60---65---70---75---80---85---90---95--100
****************************************************************************************************
Unigram tokens 12661559 types 59884
=== 2/5 Calculating and sorting adjusted counts ===
Chain sizes: 1:718608 2:24941049856 3:46764470272
Statistics:
1 59884 D1=0.997828 D2=1.35525 D3+=0.149063
2 126214 D1=0.862807 D2=0.765178 D3+=0.921723
3 151779 D1=0.67498 D2=1.05899 D3+=1.35668
Memory estimate for binary LM:
type      kB
probing 7146 assuming -p 1.5
probing 8120 assuming -r models -p 1.5
trie    3768 without quantization
trie    2621 assuming -q 8 -b 8 quantization 
trie    3602 assuming -a 22 array pointer compression
trie    2455 assuming -a 22 -q 8 -b 8 array pointer compression and quantization
=== 3/5 Calculating and sorting initial probabilities ===
Chain sizes: 1:718608 2:2019424 3:3035580
----5---10-

In [ ]:
from google.colab import files

files.download("5_gram_th.arpa")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# !./build_binary 5_gram_th.arpa.arpa 5_gram_th.binary

# Setup

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive/')

Mounted at /content/gdrive/


In [ ]:
import warnings
warnings.filterwarnings("ignore")

In [ ]:
from transformers import Wav2Vec2Processor, Wav2Vec2ForCTC

processor = Wav2Vec2Processor.from_pretrained("Your-model-folder-url-here")
model = Wav2Vec2ForCTC.from_pretrained("Your-model-folder-url-here")
model.to("cuda")

Wav2Vec2ForCTC(
  (wav2vec2): Wav2Vec2Model(
    (feature_extractor): Wav2Vec2FeatureEncoder(
      (conv_layers): ModuleList(
        (0): Wav2Vec2LayerNormConvLayer(
          (conv): Conv1d(1, 512, kernel_size=(10,), stride=(5,))
          (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (activation): GELUActivation()
        )
        (1-4): 4 x Wav2Vec2LayerNormConvLayer(
          (conv): Conv1d(512, 512, kernel_size=(3,), stride=(2,))
          (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (activation): GELUActivation()
        )
        (5-6): 2 x Wav2Vec2LayerNormConvLayer(
          (conv): Conv1d(512, 512, kernel_size=(2,), stride=(2,))
          (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (activation): GELUActivation()
        )
      )
    )
    (feature_projection): Wav2Vec2FeatureProjection(
      (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
      (projec

In [ ]:
from pyctcdecode import build_ctcdecoder

n_gram_path = 'Your-n-gram-model-folder-url-here'

vocab_dict = processor.tokenizer.get_vocab()
vocab_list = [k for k, v in sorted(vocab_dict.items(), key=lambda item: item[1])]
decoder = build_ctcdecoder(
    labels=vocab_list,
    kenlm_model_path=n_gram_path
)

# Prepare Data for test

In [ ]:
!tar -xvzf 'Your-dataset-folder-url-here' -C /content/

เอาต์พุตของการสตรีมมีการตัดเหลือเพียง 5000 บรรทัดสุดท้าย
cv-corpus-7.0-2021-07-21/th/clips/common_voice_th_27269603.mp3
cv-corpus-7.0-2021-07-21/th/clips/common_voice_th_27269604.mp3
cv-corpus-7.0-2021-07-21/th/clips/common_voice_th_27269606.mp3
cv-corpus-7.0-2021-07-21/th/clips/common_voice_th_27269608.mp3
cv-corpus-7.0-2021-07-21/th/clips/common_voice_th_27269609.mp3
cv-corpus-7.0-2021-07-21/th/clips/common_voice_th_27269611.mp3
cv-corpus-7.0-2021-07-21/th/clips/common_voice_th_27269613.mp3
cv-corpus-7.0-2021-07-21/th/clips/common_voice_th_27269615.mp3
cv-corpus-7.0-2021-07-21/th/clips/common_voice_th_27269616.mp3
cv-corpus-7.0-2021-07-21/th/clips/common_voice_th_27269617.mp3
cv-corpus-7.0-2021-07-21/th/clips/common_voice_th_27269618.mp3
cv-corpus-7.0-2021-07-21/th/clips/common_voice_th_27269619.mp3
cv-corpus-7.0-2021-07-21/th/clips/common_voice_th_27269682.mp3
cv-corpus-7.0-2021-07-21/th/clips/common_voice_th_27269683.mp3
cv-corpus-7.0-2021-07-21/th/clips/common_voice_th_27269684.mp

In [ ]:
from datasets import Dataset

test_dataset = Dataset.from_csv('Your-dataset-folder-url-here', delimiter='\t')
test_dataset

Dataset({
    features: ['path', 'sentence'],
    num_rows: 4276
})

In [ ]:
import torch
import torchaudio
import re
import evaluate
import os

In [ ]:
chars_to_ignore_regex = '[\,\?\.\!\-\;\:\"\“]'
resampler = torchaudio.transforms.Resample(48_000, 16_000)

In [ ]:
from pythainlp.tokenize import word_tokenize

def th_tokenize(batch):
    batch["sentence"] = " ".join(word_tokenize(batch["sentence"], engine="deepcut"))
    # batch["sentence"] = " ".join(word_tokenize(batch["sentence"], engine="attacut"))
    return batch

In [ ]:
# Preprocessing the datasets.
# # We need to read the audio files as arrays
def speech_file_to_array_fn(batch):
    audio_file_path = os.path.join("Your-dataset-folder-url-here", batch["path"])

    if not os.path.exists(audio_file_path):
        print(f"File does not exist: {audio_file_path}")
        return batch

    batch["sentence"] = re.sub(chars_to_ignore_regex, '', batch["sentence"]).lower()
    speech_array, sampling_rate = torchaudio.load(audio_file_path)
    batch["speech"] = resampler(speech_array).squeeze().numpy()
    return batch

In [ ]:
def predict(batch):
    inputs = processor(batch["speech"], sampling_rate=16_000, return_tensors="pt", padding=True)

    with torch.no_grad():
        logits = model(inputs.input_values.to("cuda"), attention_mask=inputs.attention_mask.to("cuda")).logits

    pred_ids = torch.argmax(logits, dim=-1)
    batch["pred_strings"] = processor.batch_decode(pred_ids)
    return batch

In [ ]:
def predict_lm(batch):
    inputs = processor(batch["speech"], sampling_rate=16_000, return_tensors="pt", padding=True)

    with torch.no_grad():
        logits = model(inputs.input_values.to("cuda"), attention_mask=inputs.attention_mask.to("cuda")).logits

    # แปลง logits เป็น list ของ logits
    logits_list = [logits[i].cpu().numpy() for i in range(logits.shape[0])]

    # ใช้ decoder ที่มีอยู่แล้ว
    pred_strings = decoder.decode_batch(logits_list=logits_list, pool=None)

    batch["pred_strings"] = pred_strings
    return batch


# Test

In [ ]:
!pip install jiwer

In [ ]:
model.freeze_feature_extractor()

In [ ]:
import evaluate

wer = evaluate.load("wer")
cer = evaluate.load("cer")

In [ ]:
# test = test_dataset.map(th_tokenize).map(speech_file_to_array_fn)
test = test_dataset.map(speech_file_to_array_fn)

In [ ]:
test

Dataset({
    features: ['path', 'sentence', 'speech'],
    num_rows: 4276
})

## Before using LM

In [ ]:
result_1 = test.map(predict, batched=True, batch_size=64, cache_file_name=None)

print("WER: {:2f}".format(100 * wer.compute(predictions=result_1["pred_strings"], references=result_1["sentence"])))
print("CER: {:2f}".format(100 * cer.compute(predictions=result_1["pred_strings"], references=result_1["sentence"])))

Map:   0%|          | 0/4276 [00:00<?, ? examples/s]

WER: 19.565552
CER: 6.348885


In [ ]:
print("ref | pred")
for obj in zip(result_1["sentence"][:15], result_1["pred_strings"]):
  print(obj)

ref | pred
('ใน เดือน กุมภาพันธ์   มัน จะ เป็น วัน ครบ รอบ ของ เรา ', 'ใน เดีืยน กุมพาพันธ์ มัน จะ เป็น วัน ครอบ ๆ ของ เรา')
('เป้าหมาย ของ เรา ก็ ไม่ ถูกต้อง ', 'เปา หมาย ของ เรา ก็ ไม่ ถูกต้อง ห')
('เธอ จะ ทำ ทุก อย่าง เท่า ที่ ทำ ได้ เพื่อ ทำ ให้ ฉัน มี ความ สุข ', 'เธอ จะ ทำ ทุกอย่าง เท่า ที่ ทำ ได้ เพื่อ ทำ ให้ ฉัน มี ความ สุข')
('เขา จัด เตรียม ถุง นอน ของ เขา เหมือน กับ กระสอบ   และ เขา ก็ มุ่งหน้า ไป ยัง ข้าง คลอง ', 'เขา จัด เตรียม ถุง นอน ของ เขา เหมือน กับ กระสอบ และ เขา ก็ มุ่งหน้า ไป ยัง ข่างคลอง')
('อ้วก ครับ   แสง สี มา พร้อม ', 'อ้วครับ แสง สี ม้า พร้อม')
('การ โจมตี ที่ เซิร์ฟเวอร์ รูท ของ พวก เรา ทำ ให้ ผู้ ดูแล ทำ งาน หนัก เกิน ไป ', 'การ โจมตี ที่ เซิร์ฟเวอร์รู้ด ของ พวก เรา ทำ ให้ ผู้ ดูแล ทำ งาน หนัก เกิน ไป')
('ตี แสก หน้า ', 'ตี ใสก หน้า')
('ค่า ใช้จ่าย ส่วน ใหญ่ ยัง เกี่ยว กับ ชีวิต ส่วน ตัว ', 'ค้า ใจ จาก  ส่วน ใหญ่ ยัง เกี่ยว กับ ชีวิต ส่วน ตัว')
('มัน ทำ ให้ ฉัน คิด ถึง ', 'มัน ทำ ให้ ฉัน คิด ถึง')
('สันติภาพ จง มี แด่ ท่าน และ พระคุณ ต่อ พระพักตร์ พระเจ้า '

## After using LM

In [ ]:
result_2 = test.map(predict_lm, batched=True, batch_size=64, cache_file_name=None)

print("WER: {:2f}".format(100 * wer.compute(predictions=result_2["pred_strings"], references=result_2["sentence"])))
print("CER: {:2f}".format(100 * cer.compute(predictions=result_2["pred_strings"], references=result_2["sentence"])))

Map:   0%|          | 0/4276 [00:00<?, ? examples/s]

WER: 22.086675
CER: 7.647497


In [ ]:
print("ref | pred")
for obj in zip(result_2["sentence"][:15], result_2["pred_strings"]):
  print(obj)

ref | pred
('ใน เดือน กุมภาพันธ์   มัน จะ เป็น วัน ครบ รอบ ของ เรา', 'ใน เดือน กลุ่ม พา พันธุ์ มัน จะ เป็น วัน ครบ อก ของ เรา')
('เป้าหมาย ของ เรา ก็ ไม่ ถูกต้อง', 'เปาหมาย ของ เรา ก็ ไม่ ถูกต้อง')
('เธอ จะ ทำ ทุก อย่าง เท่า ที่ ทำ ได้ เพื่อ ทำ ให้ ฉัน มี ความ สุข', 'เธอ จะ ทำ ทุก อย่าง เท่า ที่ ทำ ได้ เพื่อ ทำ ให้ ฉัน มี ความ สุข')
('เขา จัด เตรียม ถุง นอน ของ เขา เหมือน กับ กระสอบ   และ เขา ก็ มุ่งหน้า ไป ยัง ข้าง คลอง', 'เขา จัด เตรียม ถุง นอน ของ เขา เหมือน กับ กระสอบ และ เขา ก็ มุ่ง หน้า ไป ยัง ห้า คลอง')
('อ้วก ครับ   แสง สี มา พร้อม', 'อั้วคับ แสง สี ม้า พร้อม คับ')
('การ โจมตี ที่ เซิร์ฟเวอร์รูท ของ พวก เรา ทำ ให้ ผู้ ดูแล ทำ งาน หนัก เกิน ไป', 'การ โจมตี ที่ เซิร์ฟเวอร์รู้ด ของ พวก เรา ทำ ให้ ผู้ ดูแล ทำ งาน หนัก เกิน ไป')
('ตี แสก หน้า', 'ตี ใสก หน้า')
('ค่า ใช้จ่าย ส่วน ใหญ่ ยัง เกี่ยว กับ ชีวิต ส่วน ตัว', 'ข้า ใจ จาก ส่วน ใหญ่ ยัง เกี่ยว กับ ชีวิต ส่วน ตัว')
('มัน ทำ ให้ ฉัน คิด ถึง', 'มัน ทำ ให้ ฉัน คิด ถึง ท ทำ')
('สันติภาพ จง มี แด่ ท่าน และ พระคุณ ต่อ พระพักตร์ พระเจ้า'

## LM using new corpus

In [ ]:
result_3 = test.map(predict_lm, batched=True, batch_size=64, cache_file_name=None)

print("WER: {:2f}".format(100 * wer.compute(predictions=result_3["pred_strings"], references=result_3["sentence"])))
print("CER: {:2f}".format(100 * cer.compute(predictions=result_3["pred_strings"], references=result_3["sentence"])))

Map:   0%|          | 0/4276 [00:00<?, ? examples/s]

WER: 19.899966
CER: 7.329694


In [ ]:
print("ref | pred")
for obj in zip(result_3["sentence"][:15], result_3["pred_strings"]):
  print(obj)

ref | pred
('ใน เดือน กุมภาพันธ์   มัน จะ เป็น วัน ครบ รอบ ของ เรา', 'ใน เดือน กุมภาพันธ์ มัน จะ เป็น วัน ครบ อก ของ เรา')
('เป้าหมาย ของ เรา ก็ ไม่ ถูกต้อง', 'เปาหมาย ของ เรา ก็ ไม่ ถูกต้อง')
('เธอ จะ ทำ ทุก อย่าง เท่า ที่ ทำ ได้ เพื่อ ทำ ให้ ฉัน มี ความ สุข', 'เธอ จะ ทำ ทุก อย่าง เท่า ที่ ทำ ได้ เพื่อ ทำ ให้ ฉัน มี ความ สุข')
('เขา จัด เตรียม ถุง นอน ของ เขา เหมือน กับ กระสอบ   และ เขา ก็ มุ่งหน้า ไป ยัง ข้าง คลอง', 'เขา จัด เตรียม ถุง นอน ของ เขา เหมือน กับ กระสอบ และ เขา ก็ มุ่งหน้า ไป ยัง ห้า คลอง')
('อ้วก ครับ   แสง สี มา พร้อม', 'อ้วคับแสงสีม้า พร้อม บบ')
('การ โจมตี ที่ เซิร์ฟเวอร์รูท ของ พวก เรา ทำ ให้ ผู้ ดูแล ทำ งาน หนัก เกิน ไป', 'การ โจมตี ที่ เซิร์ฟเวอร์ รู้ด ของ พวก เรา ทำ ให้ ผู้ ดูแล ทำ งาน หนัก เกิน ไป')
('ตี แสก หน้า', 'ตี แสก หน้า')
('ค่า ใช้จ่าย ส่วน ใหญ่ ยัง เกี่ยว กับ ชีวิต ส่วน ตัว', 'ค่า ใจ จา ส่วน ใหญ่ ยัง เกี่ยว กับ ชีวิต ส่วน ตัว')
('มัน ทำ ให้ ฉัน คิด ถึง', 'มัน ทำ ให้ ฉัน คิด ถึง ท ทำ')
('สันติภาพ จง มี แด่ ท่าน และ พระคุณ ต่อ พระพักตร์ พระเจ้า', 'สันติภาพ

## LM using new model and 6 gram-new corpus

In [ ]:
result_4 = test.map(predict_lm, batched=True, batch_size=64, cache_file_name=None)

print("WER: {:2f}".format(100 * wer.compute(predictions=result_4["pred_strings"], references=result_4["sentence"])))
print("CER: {:2f}".format(100 * cer.compute(predictions=result_4["pred_strings"], references=result_4["sentence"])))

Parameter 'function'=<function predict_lm at 0x7fb4c2229ee0> of the transform datasets.arrow_dataset.Dataset._map_single couldn't be hashed properly, a random hash was used instead. Make sure your transforms and parameters are serializable with pickle or dill for the dataset fingerprinting and caching to work. If you reuse this transform, the caching mechanism will consider it to be different from the previous calls and recompute everything. This warning is only showed once. Subsequent hashing failures won't be showed.


Map:   0%|          | 0/4276 [00:00<?, ? examples/s]

WER: 21.106147
CER: 7.183664


In [ ]:
print("ref | pred")
for obj in zip(result_4["sentence"][:15], result_4["pred_strings"]):
  print(obj)

ref | pred
('ใน เดือน กุมภาพันธ์   มัน จะ เป็น วัน ครบ รอบ ของ เรา ', 'ใน เดือน กุมพาพันธ์ มัน จะ เป็น วัน ครบ ๆ ของ เรา')
('เป้าหมาย ของ เรา ก็ ไม่ ถูกต้อง ', 'เปาหมาย ของ เรา ก็ ไม่ ถูกต้อง ห')
('เธอ จะ ทำ ทุก อย่าง เท่า ที่ ทำ ได้ เพื่อ ทำ ให้ ฉัน มี ความ สุข ', 'เธอ จะ ทำ ทุก อย่าง เท่า ที่ ทำ ได้ เพื่อ ทำ ให้ ฉัน มี ความ สุข')
('เขา จัด เตรียม ถุง นอน ของ เขา เหมือน กับ กระสอบ   และ เขา ก็ มุ่งหน้า ไป ยัง ข้าง คลอง ', 'เขา จัด เตรียม ถุง นอน ของ เขา เหมือน กับ กระสอบ และ เขา ก็ มุ่งหน้า ไป ยัง ข้าง คลอง')
('อ้วก ครับ   แสง สี มา พร้อม ', 'อ้วครับ แสง สี ม้า พร้อม')
('การ โจมตี ที่ เซิร์ฟเวอร์ รูท ของ พวก เรา ทำ ให้ ผู้ ดูแล ทำ งาน หนัก เกิน ไป ', 'การ โจมตี ที่ เซิร์ฟเวอร์รู้ด ของ พวก เรา ทำ ให้ ผู้ ดูแล ทำ งาน หนัก เกิน ไป')
('ตี แสก หน้า ', 'ตีใสก หน้า')
('ค่า ใช้จ่าย ส่วน ใหญ่ ยัง เกี่ยว กับ ชีวิต ส่วน ตัว ', 'ค้า ใจ จาก ส่วน ใหญ่ ยัง เกี่ยว กับ ชีวิต ส่วน ตัว')
('มัน ทำ ให้ ฉัน คิด ถึง ', 'มัน ทำ ให้ ฉัน คิด ถึง')
('สันติภาพ จง มี แด่ ท่าน และ พระคุณ ต่อ พระพักตร์ พระเจ้า ', '

## LM using new model and 6-gram corpus

In [ ]:
result_5 = test.map(predict_lm, batched=True, batch_size=64, cache_file_name=None)

print("WER: {:2f}".format(100 * wer.compute(predictions=result_5["pred_strings"], references=result_5["sentence"])))
print("CER: {:2f}".format(100 * cer.compute(predictions=result_5["pred_strings"], references=result_5["sentence"])))

Map:   0%|          | 0/4276 [00:00<?, ? examples/s]

WER: 17.254660
CER: 6.256413


In [ ]:
print("ref | pred")
for obj in zip(result_5["sentence"][:15], result_5["pred_strings"]):
  print(obj)

ref | pred
('ใน เดือน กุมภาพันธ์   มัน จะ เป็น วัน ครบ รอบ ของ เรา ', 'ใน เดือน กุมภาพันธ์ มัน จะ เป็น วัน ครอบ ของ เรา')
('เป้าหมาย ของ เรา ก็ ไม่ ถูกต้อง ', 'เปา หมาย ของ เรา ก็ ไม่ ถูกต้อง ห')
('เธอ จะ ทำ ทุก อย่าง เท่า ที่ ทำ ได้ เพื่อ ทำ ให้ ฉัน มี ความ สุข ', 'เธอ จะ ทำ ทุกอย่าง เท่า ที่ ทำ ได้ เพื่อ ทำ ให้ ฉัน มี ความ สุข')
('เขา จัด เตรียม ถุง นอน ของ เขา เหมือน กับ กระสอบ   และ เขา ก็ มุ่งหน้า ไป ยัง ข้าง คลอง ', 'เขา จัด เตรียม ถุง นอน ของ เขา เหมือน กับ กระสอบ และ เขา ก็ มุ่งหน้า ไป ยัง ข้าง คลอง')
('อ้วก ครับ   แสง สี มา พร้อม ', 'อ้วครับ แสง สี ม้า พร้อม')
('การ โจมตี ที่ เซิร์ฟเวอร์ รูท ของ พวก เรา ทำ ให้ ผู้ ดูแล ทำ งาน หนัก เกิน ไป ', 'การ โจมตี ที่ เซิร์ฟเวอร์ รูด ของ พวก เรา ทำ ให้ ผู้ ดูแล ทำ งาน หนัก เกิน ไป')
('ตี แสก หน้า ', 'ตี แสก หน้า')
('ค่า ใช้จ่าย ส่วน ใหญ่ ยัง เกี่ยว กับ ชีวิต ส่วน ตัว ', 'ค้า ใจ จาก ส่วน ใหญ่ ยัง เกี่ยว กับ ชีวิต ส่วน ตัว')
('มัน ทำ ให้ ฉัน คิด ถึง ', 'มัน ทำ ให้ ฉัน คิด ถึง')
('สันติภาพ จง มี แด่ ท่าน และ พระคุณ ต่อ พระพักตร์ พระเจ้า ', '

## LM using new model and 3-gram corpus

In [ ]:
result_6 = test.map(predict_lm, batched=True, batch_size=64, cache_file_name=None)

print("WER: {:2f}".format(100 * wer.compute(predictions=result_6["pred_strings"], references=result_6["sentence"])))
print("CER: {:2f}".format(100 * cer.compute(predictions=result_6["pred_strings"], references=result_6["sentence"])))

Parameter 'function'=<function predict_lm at 0x7c2bf68c8cc0> of the transform datasets.arrow_dataset.Dataset._map_single couldn't be hashed properly, a random hash was used instead. Make sure your transforms and parameters are serializable with pickle or dill for the dataset fingerprinting and caching to work. If you reuse this transform, the caching mechanism will consider it to be different from the previous calls and recompute everything. This warning is only showed once. Subsequent hashing failures won't be showed.


Map:   0%|          | 0/4276 [00:00<?, ? examples/s]

WER: 17.276229
CER: 6.259580


In [ ]:
print("ref | pred")
for obj in zip(result_6["sentence"][:15], result_6["pred_strings"]):
  print(obj)

ref | pred
('ใน เดือน กุมภาพันธ์   มัน จะ เป็น วัน ครบ รอบ ของ เรา ', 'ใน เดือน กุมภาพันธ์ มัน จะ เป็น วัน ครอบ ของ เรา')
('เป้าหมาย ของ เรา ก็ ไม่ ถูกต้อง ', 'เปา หมาย ของ เรา ก็ ไม่ ถูกต้อง ห')
('เธอ จะ ทำ ทุก อย่าง เท่า ที่ ทำ ได้ เพื่อ ทำ ให้ ฉัน มี ความ สุข ', 'เธอ จะ ทำ ทุกอย่าง เท่า ที่ ทำ ได้ เพื่อ ทำ ให้ ฉัน มี ความ สุข')
('เขา จัด เตรียม ถุง นอน ของ เขา เหมือน กับ กระสอบ   และ เขา ก็ มุ่งหน้า ไป ยัง ข้าง คลอง ', 'เขา จัด เตรียม ถุง นอน ของ เขา เหมือน กับ กระสอบ และ เขา ก็ มุ่งหน้า ไป ยัง ข้าง คลอง')
('อ้วก ครับ   แสง สี มา พร้อม ', 'อ้วครับ แสง สี ม้า พร้อม')
('การ โจมตี ที่ เซิร์ฟเวอร์ รูท ของ พวก เรา ทำ ให้ ผู้ ดูแล ทำ งาน หนัก เกิน ไป ', 'การ โจมตี ที่ เซิร์ฟเวอร์ รูด ของ พวก เรา ทำ ให้ ผู้ ดูแล ทำ งาน หนัก เกิน ไป')
('ตี แสก หน้า ', 'ตี แสก หน้า')
('ค่า ใช้จ่าย ส่วน ใหญ่ ยัง เกี่ยว กับ ชีวิต ส่วน ตัว ', 'ค้า ใจ จาก ส่วน ใหญ่ ยัง เกี่ยว กับ ชีวิต ส่วน ตัว')
('มัน ทำ ให้ ฉัน คิด ถึง ', 'มัน ทำ ให้ ฉัน คิด ถึง')
('สันติภาพ จง มี แด่ ท่าน และ พระคุณ ต่อ พระพักตร์ พระเจ้า ', '